In [ ]:
class FeatureFusionModule(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.fusion_conv = nn.Conv2d(in_channels=channels * 2, out_channels=channels, kernel_size=1, stride=1)

    def forward(self, x1, x2):
        # x1 和 x2 都是相同的维度，我们将它们在通道维度上拼接
        x = torch.cat((x1, x2), dim=1)
        x = self.fusion_conv(x)
        return x

class ImageEncoderViT(nn.Module):
    def __init__(
        self,
        args,
        img_size: int = 1024,
        patch_size: int = 16,
        in_chans: int = 3,
        embed_dim: int = 768,
        depth: int = 12,
        num_heads: int = 12,
        mlp_ratio: float = 4.0,
        out_chans: int = 256,
        qkv_bias: bool = True,
        norm_layer: Type[nn.Module] = nn.LayerNorm,
        act_layer: Type[nn.Module] = nn.GELU,
        use_abs_pos: bool = True,
        use_rel_pos: bool = False,
        rel_pos_zero_init: bool = True,
        window_size: int = 0,
        global_attn_indexes: Tuple[int, ...] = (),
    ):
        super().__init__()
        self.args = args
        self.MyFF = FFParser(3, 1024, 1024)
        self.patch_embed = PatchEmbed(
            kernel_size=(patch_size, patch_size),
            stride=(patch_size, patch_size),
            in_chans=in_chans,
            embed_dim=embed_dim,
        )

        if use_abs_pos:
            self.pos_embed = nn.Parameter(
                torch.zeros(1, img_size // patch_size, img_size // patch_size, embed_dim)
            )

        self.feature_fusion_modules = nn.ModuleList([FeatureFusionModule(embed_dim) for _ in range(depth)])
        self.blocks = nn.ModuleList([
            Block(
                args=self.args,
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                norm_layer=norm_layer,
                act_layer=act_layer,
                use_rel_pos=use_rel_pos,
                rel_pos_zero_init=rel_pos_zero_init,
                window_size=window_size if i not in global_attn_indexes else 0,
            ) for i in range(depth)
        ])

        self.neck = nn.Sequential(
            nn.Conv2d(embed_dim, out_chans, kernel_size=1, bias=False),
            LayerNorm2d(out_chans),
            nn.Conv2d(out_chans, out_chans, kernel_size=3, padding=1, bias=False),
            LayerNorm2d(out_chans),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        high_freq = self.MyFF(x)
        patch_out = self.patch_embed(x)

        # First fusion and input to the first Transformer Block
        fused_feature_1 = self.feature_fusion_modules[0](patch_out, high_freq)
        x = torch.cat([patch_out, fused_feature_1], dim=1)
        x = self.blocks[0](x)
        if self.pos_embed is not None:
            x += self.pos_embed

        # Loop through subsequent blocks, fusing output of previous block with its corresponding fusion module output
        for i in range(1, len(self.blocks)):
            previous_block_output = x
            x = self.feature_fusion_modules[i](previous_block_output, patch_out)  # Fusion of current block input and patch embedding output
            x = self.blocks[i](x)  # Current block processing

        x = self.neck(x.permute(0, 3, 1, 2))  # Final processing in 'neck'
        return x
